# AE: 2次元への圧縮と復元 / AE: Compression to two dimensions and reconstruction

講義対応：非線形次元削減，潜在表現，再構成誤差．

Lecture connection: nonlinear dimensionality reduction, latent representations, and reconstruction error.

**Google Colabでの使い方**：この `.ipynb` をアップロードし，「ランタイム → すべてのセルを実行」します．CPUで実行できます．

**Using Google Colab**: Upload this `.ipynb` and select Runtime → Run all. A CPU is sufficient.

タイトル，コメント，説明，表示ラベルは日本語・英語併記です．各ノートブックは独立して動きます．乱数シードを固定しますが，GPUやライブラリの違いで結果は多少変わります．

Titles, comments, explanations, and display labels are provided in Japanese and English. Each notebook runs independently. Seeds are fixed, but results can vary across GPUs and library versions.


64画素を2個の数値へ圧縮します．$z=f_\theta(x)$，$\hat x=g_\phi(z)$，$L=\|x-\hat x\|^2$．学習に数字ラベルは使いません．

Compress 64 pixels into two numbers: $z=f_\theta(x)$, $\hat x=g_\phi(z)$, $L=\|x-\hat x\|^2$. Digit labels are not used for training.


## 準備 / Setup


In [ ]:
import sys, subprocess, importlib.util
# Colabの既存PyTorchを優先し，不足パッケージだけ追加します． / Reuse Colab packages; install only missing dependencies.
for module, package in [('torch','torch'), ('sklearn','scikit-learn'), ('ipywidgets','ipywidgets')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
import random, math, time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
import ipywidgets as widgets
from IPython.display import display, clear_output
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.set_num_threads(2)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams.update({'figure.dpi':110, 'font.size':11, 'axes.spines.top':False, 'axes.spines.right':False})
print('PyTorch:', torch.__version__, '| 実行機器 / Device:', device)


# 日本語フォントを設定します． / Configure a Japanese font for bilingual figures.
if importlib.util.find_spec('japanize_matplotlib') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'japanize-matplotlib'])
import japanize_matplotlib
plt.rcParams['axes.unicode_minus'] = False


## 設定 / Configuration


In [ ]:
BATCH_SIZE=128
LATENT_DIM=2
EPOCHS=45
LEARNING_RATE=.002


## データ / Data


In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
raw = load_digits()
# sklearn同梱の8×8手書き数字．通信・認証は不要です． / Use the bundled sklearn 8×8 digits; no dataset download or authentication.
# 画像モデルの学習にクラスラベルは使いません．可視化だけに使います． / Class labels are used for visualization, not image-model training.
images = (raw.data / 16.0).astype('float32')
train_idx, test_idx = train_test_split(np.arange(len(images)), test_size=0.2,
                                    random_state=SEED, stratify=raw.target)
x_train = torch.tensor(images[train_idx])
x_test = torch.tensor(images[test_idx])
y_test = raw.target[test_idx]
loader = DataLoader(TensorDataset(x_train), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
def image_grid(x, title='', nrow=8):
    x = torch.as_tensor(x).detach().cpu().reshape(-1,8,8).numpy()
    rows = math.ceil(len(x)/nrow)
    fig, ax = plt.subplots(rows,nrow,figsize=(nrow,rows*1.1),squeeze=False)
    for a in ax.ravel(): a.axis('off')
    for a,im in zip(ax.ravel(),x): a.imshow(im,cmap='gray',vmin=0,vmax=1)
    fig.suptitle(title); plt.tight_layout(); plt.show()
image_grid(x_train[:16], '訓練画像（8×8画素） / Training examples (8×8 pixels)')
print('訓練数 / Train:',len(x_train),'| 評価数 / Held-out:',len(x_test))


## モデル / Model


In [ ]:
class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder=nn.Sequential(nn.Linear(64,128),nn.ReLU(),nn.Linear(128,32),nn.ReLU(),nn.Linear(32,LATENT_DIM))
        self.decoder=nn.Sequential(nn.Linear(LATENT_DIM,32),nn.ReLU(),nn.Linear(32,128),nn.ReLU(),nn.Linear(128,64),nn.Sigmoid())
    def forward(self,x):return self.decoder(self.encoder(x))
model=Autoencoder().to(device);optimizer=torch.optim.Adam(model.parameters(),lr=LEARNING_RATE)


## 学習 / Training


In [ ]:
history=[]
for epoch in range(EPOCHS):
    model.train();total=0
    for (xb,) in loader:
        xb=xb.to(device);optimizer.zero_grad();loss=F.mse_loss(model(xb),xb)
        loss.backward();optimizer.step();total+=loss.item()*len(xb)
    model.eval()
    with torch.no_grad():v=F.mse_loss(model(x_test.to(device)),x_test.to(device)).item()
    history.append((total/len(x_train),v))
    if epoch==0 or (epoch+1)%10==0:print(f'エポック / Epoch {epoch+1}: 訓練 / train={history[-1][0]:.4f}, 評価 / held-out={v:.4f}')
plt.plot(history);plt.legend(['訓練 / Train','評価 / Held-out']);plt.title('再構成MSE / Reconstruction MSE');plt.show()


## 評価と可視化 / Evaluation and visualization


In [ ]:
model.eval()
with torch.no_grad():
    z=model.encoder(x_test.to(device)).cpu();reconstruction=model(x_test.to(device)).cpu()
image_grid(torch.cat([x_test[:8],reconstruction[:8]]),'原画像と復元画像（各行）\nHeld-out originals (top) / Reconstructions (bottom)')
fig,ax=plt.subplots(figsize=(6,5));sc=ax.scatter(z[:,0],z[:,1],c=y_test,cmap='tab10',s=14)
fig.colorbar(sc,ax=ax,ticks=range(10),label='数字（表示専用） / Digit (visualization only)')
ax.set_xlabel('z1');ax.set_ylabel('z2');plt.show()
print('潜在表現の形状 / Latent shape:',tuple(z.shape))


## インタラクティブ操作 / Interactive controls


In [ ]:
zmin=z.quantile(.02,dim=0).numpy();zmax=z.quantile(.98,dim=0).numpy()
def decode_point(z1=0.,z2=0.):
    model.eval()
    point=z.median(dim=0).values.to(device).unsqueeze(0).clone()
    point[0,0]=z1;point[0,1]=z2
    with torch.no_grad():im=model.decoder(point).cpu().reshape(8,8)
    plt.figure(figsize=(2.5,2.5));plt.imshow(im,cmap='gray',vmin=0,vmax=1);plt.title(f'z=({z1:.2f},{z2:.2f})');plt.axis('off');plt.show()
widgets.interact(decode_point,
    z1=widgets.FloatSlider(min=float(zmin[0]),max=float(zmax[0]),value=float(z[:,0].median()),step=.1,description='z1', style={'description_width':'initial'}, layout=widgets.Layout(width='auto',min_width='260px'),continuous_update=False),
    z2=widgets.FloatSlider(min=float(zmin[1]),max=float(zmax[1]),value=float(z[:,1].median()),step=.1,description='z2', style={'description_width':'initial'}, layout=widgets.Layout(width='auto',min_width='260px'),continuous_update=False))


## 観察すること / What to observe
- 2次元化では細部が失われることがあります．LATENT_DIMを8に変えると，2D可視化は先頭2成分のみになります．スライダーでは残りの成分を中央値に固定します．潜在次元は2以上にしてください．
  Compression to two dimensions may lose detail. With LATENT_DIM = 8, the scatter plot shows only the first two components; sliders hold the other components at their medians. Use at least two latent dimensions.
- 通常のAEでは，潜在空間の任意の点が自然な数字を作るとは限りません．
  An arbitrary latent point in a standard AE need not decode to a natural-looking digit.
- 次のVAEでは，潜在分布を事前分布に近づける学習を加えます．
  The next VAE demo adds a training term that brings latent distributions closer to a prior.


## 参考 / References
- [PyTorch documentation](https://docs.pytorch.org/docs/stable/nn.html)
- [scikit-learn: digits dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html)
- [Colab widget example](https://colab.research.google.com/notebooks/snippets/advanced_outputs.ipynb)
